<a href="https://colab.research.google.com/github/bjanesko/ComputationalChemistryForExperimentalists/blob/main/Chapter12_Solvent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Computational Chemistry for Experimentalists
## Chapter 12: Solvent

Our calculations up to now have treated single isolated molecules, but most real chemistry of interest occurs in solvent. Modeling solvent is hard: naive models require conformational sampling of huge numbers of solvent molecules for each solute. Electronic structure calculations typically use continuum solvent models, replacing the details of the real solvent with a featureless dielectric continuum surrounding a solute molecule. Adding a few real solvent molecules inside the dielectric cavity can compensate for some limitations of the continuum model. We'll explore both kinds of models for some critical problems in solvent chemistry, including the dissociation of salt and water.

In [ ]:
!pip install pyscf
!pip install py3Dmol
!pip install rdkit
!pip install geometric
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem import AllChem
from pyscf import gto,scf,dft,solvent
from pyscf.tools import cubegen
import py3Dmol
import numpy
import matplotlib.pyplot as plt
from pyscf.geomopt.geometric_solver import optimize
from rdkit.Geometry import Point3D
from rdkit.Chem import rdDetermineBonds

## Part 1: Ions

Table salt NaCl(s) is  the prototypical example of an ionic compound and ions in solution. However, in the gas phase, a single formula unit of NaCl will dissociate into radicals, not ions! Experimentally, the gas-phase ionization potential of Na is 5.139 eV, the gas-phase electron affinity of Cl is 3.613 eV, making electron transfer from Na to Cl uphill, reaction energy 1.526 eV.  This example shows how continuum solvent molecules capture this effect. We'll choose a sign convention where negative relative energies mean ions are more stable, positive relative energies mean radicals are more stable


In [ ]:
basis='def2svp'
def do_NaCl(solvent=None):
  mna=gto.Mole(atom='Na',basis=basis,spin=1).build()
  mnap=gto.Mole(atom='Na',basis=basis,charge=1).build()
  mcl=gto.Mole(atom='Cl',basis=basis,spin=1).build()
  mclm=gto.Mole(atom='Cl',basis=basis,charge=-1).build()
  scfs=[dft.UKS(mna,xc='b3lyp'),dft.UKS(mnap,xc='b3lyp'),dft.UKS(mcl,xc='b3lyp'),dft.UKS(mclm,xc='b3lyp')]
  es=[]
  if solvent is None:
    for s in scfs:
      e=s.kernel()
      print("!!!",e)
      es.append(e)
  else:
    if solvent == 'PCM':
      scfs=[dft.UKS(mna,xc='b3lyp').PCM(),dft.UKS(mnap,xc='b3lyp').PCM(),dft.UKS(mcl,xc='b3lyp').PCM(),dft.UKS(mclm,xc='b3lyp').PCM()]
    if solvent == 'SMD':
      scfs=[dft.UKS(mna,xc='b3lyp').SMD(),dft.UKS(mnap,xc='b3lyp').SMD(),dft.UKS(mcl,xc='b3lyp').SMD(),dft.UKS(mclm,xc='b3lyp').SMD()]
    if solvent == 'ddCOSMO':
      scfs=[dft.UKS(mna,xc='b3lyp').ddCOSMO(),dft.UKS(mnap,xc='b3lyp').ddCOSMO(),dft.UKS(mcl,xc='b3lyp').ddCOSMO(),dft.UKS(mclm,xc='b3lyp').ddCOSMO()]
    for s in scfs:
      if solvent == 'PCM':
        s.with_solvent.method="IEF-PCM"
      s.with_solvent.eps=78.3553
      e=s.kernel()
      print("!!!",e)
      es.append(e)


  de=27.211*( (es[1]+es[3])-(es[0]+es[2])) # (NA(+) + Cl(-)) - (Na + Cl)
  return(de)


deg = do_NaCl()
dew=do_NaCl(solvent='PCM')
des=do_NaCl(solvent='SMD')
dec=do_NaCl(solvent='ddCOSMO')
print('==== RESULTS ======')
print('Gas phase DE: %.3f eV '%(deg))
print('PCM water DE: %.3f eV '%(dew))
print('SMD water DE: %.3f eV '%(des))
print('ddCOSMO water DE: %.3f eV '%(dew))

## Part 2: Explicit solvent

Sometimes treating explicit solvent molecules is essential for capturing solvent chemistry. An example is the dissociation of water H2O --> H(+) + OH(-), where the H(+) must exist as hydronium ion. Here we compare the water dissociation constant pKW computed with SMD continuum solvent and zero vs one explicit solvent molecules. We'll see that explicit solvent is critical to getting anywhere near the experimental value 14. I use M06-2Z/cc-pvtz geometries from the CCCBDB



In [ ]:
basis='def2svp'
mhp=gto.Mole(atom='H',charge=1,basis=basis).build()
mh2o=gto.Mole(atom='O 0.0000	0.0000	0.1170;H	0.0000	0.7599	-0.4679;	 H 0.0000	-0.7599	-0.4679',basis=basis).build()
mh3op=gto.Mole(atom='O	0.0000	0.0000	0.0706;H	0.0000	0.9441	-0.1884;H	0.8177	-0.4721	-0.1884;H	-0.8177	-0.4721	-0.1884',charge=1,basis=basis).build()
mohm=gto.Mole(atom='O	0.0000	0.0000	0.1074;H	0.0000	0.0000	-0.8593',charge=-1,basis=basis).build()

mols=[mh2o,mohm,mhp,mh3op]
es=[]
for m in mols:
  mf=dft.UKS(m,xc='b3lyp').SMD()
  mf.with_solvent.eps=78.3553
  e=mf.kernel()
  print("!!!",e)
  es.append(e)
de1=2625.5*(es[2]+es[1]-es[0]) # H+ + OH(-) - H2O
de2=2625.5*(es[3]+es[1]-2*es[0]) # H3O+ + OH(-) - 2 H2O
kw1=-1.0*de1/(2.303*0.008314*298)
kw2=-1.0*de2/(2.303*0.008314*298)
print('==== RESULTS ==== ')
print('No explicit water: %.0f '%(de1))
print('One explicit water: %.0f '%(de2))
print('Experiment: 14 ')

## Practice Problems

Part 1: Compute the water dissociation constant pKw using more than one explicit solvent molecule. How close can you get to experiment?

Part 2: Compute the pKa of acetic acid in the gas phase and in water. How close can you get to the experimental value in water of 4.76? How close can you get to the experimental gas-phase value of about -250 (deprotonation free energy 1430 kJ/mol)?